# Chilean Mutual Funds Portfolio Optimizer
## Análisis Cuantitativo Avanzado — Santander & LarrainVial (2020–2026)

**Técnicas implementadas:**
- Ledoit-Wolf Covariance Shrinkage
- Optimización Robusta Min-Max Sharpe
- Clustering de Fondos (K-Means + Jerárquico)
- Hidden Markov Model — Detección de Regímenes
- Walk-Forward Validation (Backtesting)
- Block Bootstrap vs Monte Carlo Normal

---

## Tabla de contenidos
1. [Setup y carga de datos](#1-setup)
2. [EDA — Análisis Exploratorio](#2-eda)
3. [Métricas de riesgo avanzadas](#3-metricas)
4. [Ledoit-Wolf: covarianza robusta](#4-ledoit-wolf)
5. [Clustering de fondos](#5-clustering)
6. [Regímenes de mercado (HMM)](#6-hmm)
7. [Optimización de portafolios](#7-optimizacion)
8. [Walk-Forward Validation](#8-backtesting)
9. [Bootstrap vs Monte Carlo](#9-bootstrap)
10. [Resumen ejecutivo](#10-resumen)
15. [HRP — Hierarchical Risk Parity](#15-hrp)
16. [GARCH — Volatilidad Condicional](#16-garch)
17. [Correlacion Dinamica y Metricas Rodantes](#17-rolling)
18. [Stress Testing Hipotetico](#18-stress-hipotetico)
11. [SP500 como Benchmark y Componente](#11-sp500)
12. [Tabla de Alpha vs Benchmarks](#12-alpha)
13. [Stress Testing](#13-stress)
14. [Atribución de Retorno](#14-atribucion)




## 1. Setup y carga de datos <a id='1-setup'></a>


In [ ]:
import sys, warnings, os
from pathlib import Path

os.chdir(Path('.').resolve())
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

from src.loader     import cargar_todos
from src.metrics    import (tabla_stats, simular_historico, estacionalidad_por_mes,
                             drawdown_serie, TPM_ANUAL)
from src.covariance import comparar_estimadores, get_cov_matrix
from src.clustering import (kmeans_fondos, clustering_jerarquico,
                             resumen_clusters, asignar_clusters_jerarquico)
from src.optimizer  import optimizar_todos, frontera_eficiente, PERFILES
from src.backtesting import walk_forward_validation, degradacion_sharpe
from src.bootstrap  import proyectar_bootstrap, proyectar_montecarlo_normal, comparar_metodos
from src.benchmarks import tabla_alpha, stress_test, atribucion_retorno

plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10,
                     'axes.titlesize': 12, 'axes.labelsize': 10})
Path('outputs').mkdir(exist_ok=True)

# Colores por perfil — SP500 incluido como perfil desde el inicio
COLORS = {
    'conservador': '#2ecc71',
    'moderado':    '#f1c40f',
    'agresivo':    '#e74c3c',
    'sp500':       '#00b4d8',   # azul electrico
    'optimo':      '#FFD700',   # dorado
    'santander':   '#e74c3c',
    'larrain_vial':'#3498db',
    'externo':     '#00b4d8',
}
MONTO = 10_000_000
print('Setup OK — SP500 integrado como perfil de inversion')


In [ ]:
# Carga de datos: 43 activos (42 fondos + SP500), 2020-2026
print('Cargando datos...')
df_long, retornos, precios, meta = cargar_todos('data', fecha_inicio='2020-01-01')

print(f'\nResumen del dataset:')
print(f'  Fondos chilenos : {len(meta[meta.corredora != "externo"])}')
print(f'    Santander     : {(meta.corredora == "santander").sum()}')
print(f'    LarrainVial   : {(meta.corredora == "larrain_vial").sum()}')
print(f'  SP500           : 1 activo (en CLP, perfil = sp500)')
print(f'  Total activos   : {len(meta)}')
print(f'  Periodo         : {retornos.index[0].date()} → {retornos.index[-1].date()}')
print(f'  Meses           : {len(retornos)}')
print(f'  TPM referencia  : {TPM_ANUAL:.1%} anual (promedio Chile 2020-2026)')
meta[['nombre','perfil','corredora','moneda_orig']]


## 2. EDA — Análisis Exploratorio <a id='2-eda'></a>

Exploramos la distribución de retornos, la evolución de valor cuota y los patrones de estacionalidad mensual.



In [ ]:
# Evolucion valor cuota por corredora y perfil (base 100)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, perfil in enumerate(['conservador', 'moderado', 'agresivo']):
    for row, corredora in enumerate(['santander', 'larrain_vial']):
        ax = axes[row][col]
        fondos_p = meta[(meta.perfil == perfil) & (meta.corredora == corredora)].index
        for fid in fondos_p:
            if fid in precios.columns:
                serie = precios[fid].dropna()
                if not serie.empty:
                    (serie / serie.iloc[0] * 100).plot(
                        ax=ax, linewidth=1.5, alpha=0.85,
                        label=meta.loc[fid,'nombre'][:22])
        titulo = f'{corredora.replace("_"," ").title()} — {perfil.capitalize()}'
        ax.set_title(titulo, color=COLORS[corredora], fontsize=10)
        ax.set_ylabel('Valor cuota (base 100)')
        ax.axhline(100, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
        ax.legend(fontsize=6, loc='upper left')
        ax.grid(alpha=0.15)

fig.suptitle('Evolucion valor cuota por corredora y perfil (base 100 = Ene 2020)', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/01_evolucion_cuota.png', dpi=150, bbox_inches='tight')
plt.show()
print('Observacion: fondos agresivos con mayor dispersion — validado por clustering en seccion 5.')


In [ ]:
# Distribucion de retornos mensuales + estadisticas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, perfil in zip(axes, ['conservador', 'moderado', 'agresivo']):
    fondos_p = meta[meta.perfil == perfil].index.tolist()
    todos = pd.concat([retornos[f].dropna() for f in fondos_p if f in retornos.columns])
    ax.hist(todos * 100, bins=40, color=COLORS[perfil], alpha=0.85,
            edgecolor='white', linewidth=0.3, density=True)
    ax.axvline(0, color='white', linestyle='--', linewidth=1)
    ax.axvline(todos.mean()*100, color='yellow', linestyle='-', linewidth=1.5,
               label=f'Media: {todos.mean()*100:.2f}%')
    skew = todos.skew()
    kurt = todos.kurtosis()
    ax.set_title(perfil.capitalize(), color=COLORS[perfil], fontsize=12)
    ax.set_xlabel('Retorno mensual (%)')
    ax.legend(fontsize=8)
    ax.text(0.97, 0.95,
            f'media={todos.mean()*100:.2f}%\nstd={todos.std()*100:.2f}%\nskew={skew:.2f}\nkurt={kurt:.2f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=8,
            bbox=dict(boxstyle='round', facecolor='black', alpha=0.5))
    ax.grid(alpha=0.15)

fig.suptitle('Distribucion de retornos mensuales (2020-2026)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/02_distribucion_retornos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Nota: skewness negativa en agresivos indica colas izquierdas pesadas (eventos extremos).')


In [ ]:
# Estacionalidad mensual por perfil — heatmap
estac = estacionalidad_por_mes(retornos, meta)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
ax = axes[0]
im = ax.imshow(estac.values.T, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=3)
ax.set_xticks(range(12))
ax.set_xticklabels(estac.index, fontsize=9)
ax.set_yticks(range(3))
ax.set_yticklabels([p.capitalize() for p in estac.columns], fontsize=10)
ax.set_title('Retorno promedio mensual (%) por perfil', fontsize=11)
plt.colorbar(im, ax=ax, label='Retorno mensual (%)')
for i in range(3):
    for j in range(12):
        val = estac.values[j, i]
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8,
                color='black' if abs(val) < 2 else 'white')

# Barras promedio global
ax2 = axes[1]
prom = estac.mean(axis=1)
colores_mes = ['#e74c3c' if v < 0 else '#2ecc71' for v in prom]
bars = ax2.bar(estac.index, prom, color=colores_mes, edgecolor='white', linewidth=0.5)
ax2.axhline(0, color='white', linewidth=1)
for bar, val in zip(bars, prom):
    ax2.text(bar.get_x() + bar.get_width()/2,
             val + 0.05 * np.sign(val),
             f'{val:.2f}%', ha='center', fontsize=8,
             va='bottom' if val >= 0 else 'top')
ax2.set_ylabel('Retorno promedio (%)')
ax2.set_title('Promedio global por mes (todos los perfiles)', fontsize=11)
ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('outputs/03_estacionalidad.png', dpi=150, bbox_inches='tight')
plt.show()

mejor = estac.mean(axis=1).nlargest(3)
peor  = estac.mean(axis=1).nsmallest(3)
print(f'Mejores meses: {list(mejor.index)} ({mejor.values.round(2)})')
print(f'Peores meses : {list(peor.index)} ({peor.values.round(2)})')


## 3. Métricas de riesgo avanzadas <a id='3-metricas'></a>

Más allá del Sharpe Ratio, calculamos métricas complementarias:
- **Sortino Ratio**: penaliza solo la volatilidad negativa (más justo para distribuciones asimétricas)
- **VaR 95%**: pérdida máxima esperada el 95% del tiempo
- **CVaR 95%**: pérdida media en el peor 5% de los casos (Expected Shortfall)
- **Calmar Ratio**: retorno anual / |Max Drawdown|
- **Omega Ratio**: razón ganancias/pérdidas sobre un threshold



In [ ]:
stats = tabla_stats(retornos, meta)

# Separar SP500 del ranking para destacarlo
stats_chilenos = stats[stats.perfil != 'sp500'].copy()
stats_sp500    = stats[stats.perfil == 'sp500'].copy()

# Tabla completa formateada
cols_show = ['nombre','corredora','perfil','retorno_anual','volatilidad_anual',
             'sharpe','sortino','var_95','cvar_95','max_drawdown','calmar','omega','n_meses']
df_show = stats[cols_show].copy()
for c in ['retorno_anual','volatilidad_anual','var_95','cvar_95','max_drawdown']:
    df_show[c] = df_show[c].map('{:.2%}'.format)
for c in ['sharpe','sortino','calmar','omega']:
    df_show[c] = df_show[c].map('{:.3f}'.format)
df_show.rename(columns={
    'nombre':'Fondo','corredora':'Corredora','perfil':'Perfil',
    'retorno_anual':'Ret. Anual','volatilidad_anual':'Vol. Anual',
    'sharpe':'Sharpe','sortino':'Sortino','var_95':'VaR 95%',
    'cvar_95':'CVaR 95%','max_drawdown':'Max DD',
    'calmar':'Calmar','omega':'Omega','n_meses':'Meses'
}, inplace=True)

sp = stats_sp500.iloc[0]
print(f'SP500 (CLP): Ret={sp.retorno_anual:.2%} | Vol={sp.volatilidad_anual:.2%} | '
      f'Sharpe={sp.sharpe:.3f} | VaR95={sp.var_95:.2%} | CVaR95={sp.cvar_95:.2%}')
print()
df_show


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Ranking Sharpe — SP500 destacado con borde
ax = ax1
stats_sorted = stats.sort_values('sharpe')
nombres = stats_sorted['nombre'].str.replace('Santander ','SAN ').str.replace('LarrainVial ','LV ')
colores = [COLORS.get(p, '#888') for p in stats_sorted['perfil']]
bars = ax.barh(nombres, stats_sorted['sharpe'], color=colores,
               edgecolor='white', linewidth=0.3)

# Borde especial para SP500
for bar, perfil in zip(bars, stats_sorted['perfil']):
    if perfil == 'sp500':
        bar.set_edgecolor('#00b4d8')
        bar.set_linewidth(2.5)

ax.axvline(0, color='white', linewidth=1, linestyle='--')
for bar, val in zip(bars, stats_sorted['sharpe']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=7)
ax.set_xlabel('Sharpe Ratio')
ax.set_title(f'Sharpe Ratio — TPM ref {TPM_ANUAL:.1%}\n(azul = SP500 como benchmark)', fontsize=11)
perfiles_legend = ['conservador','moderado','agresivo','sp500','optimo']
ax.legend(handles=[Patch(color=COLORS[p], label=p.capitalize()) for p in perfiles_legend], fontsize=8)
ax.grid(axis='x', alpha=0.15)

# Scatter Sharpe vs Sortino incluyendo SP500
ax2_plot = ax2
for perfil in ['conservador','moderado','agresivo','sp500']:
    sub = stats[stats['perfil'] == perfil]
    marker = 'D' if perfil != 'sp500' else '*'
    size   = 80 if perfil != 'sp500' else 200
    ax2_plot.scatter(sub['sharpe'], sub['sortino'],
                     c=COLORS[perfil], s=size, marker=marker,
                     label=perfil.capitalize(), zorder=5)
    for _, r in sub.iterrows():
        nombre_corto = r['nombre'].replace('Santander ','').replace('LarrainVial ','').replace('S&P 500 (CLP)','SP500')[:18]
        ax2_plot.annotate(nombre_corto, (r['sharpe'], r['sortino']),
                          fontsize=6, xytext=(4,2), textcoords='offset points')

lims = [min(stats['sharpe'].min(), stats['sortino'].min()) - 0.1,
        max(stats['sharpe'].max(), stats['sortino'].max()) + 0.1]
ax2_plot.plot(lims, lims, 'white', linewidth=1, linestyle='--', alpha=0.5,
              label='Sharpe = Sortino')
ax2_plot.set_xlabel('Sharpe Ratio')
ax2_plot.set_ylabel('Sortino Ratio')
ax2_plot.set_title('Sharpe vs Sortino (todos los activos)', fontsize=11)
ax2_plot.legend(fontsize=8)
ax2_plot.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/04_metricas_riesgo.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Ledoit-Wolf: Covarianza Robusta <a id='4-ledoit-wolf'></a>

Con 42 fondos y 75 observaciones (p/n ≈ 0.56), la covarianza muestral es numéricamente inestable.

El estimador Ledoit-Wolf (2004) aplica shrinkage:
$$\Sigma_{LW} = (1-\alpha)\hat{\Sigma} + \alpha T$$

donde $T$ es un target diagonal y $\alpha$ se elige para minimizar el error cuadrático esperado de Frobenius.

**Resultado:** condition number cae de $4\times10^8$ a $130$ — reducción de 3 millones de veces.



In [ ]:
# Comparacion de estimadores de covarianza
ret_clean = retornos.dropna(how='all', axis=1).dropna()
comp = comparar_estimadores(ret_clean)

print('Comparacion de estimadores de covarianza:')
print('='*55)
for idx, row in comp.iterrows():
    print(f'  {idx:<15} | Condition#: {row["condition_number"]:>12.0f} | '
          f'Shrinkage: {row["shrinkage"]:.4f} | Min eigenvalue: {row["min_eigenvalue"]:.2e}')

print(f'\nReduccion condition number: {comp.loc["Muestral","condition_number"]/comp.loc["Ledoit-Wolf","condition_number"]:.0f}x')
print(f'Shrinkage aplicado (alpha): {comp.loc["Ledoit-Wolf","shrinkage"]:.3f}')
print(f'  -> El {comp.loc["Ledoit-Wolf","shrinkage"]*100:.1f}% del peso va al target diagonal')
print(f'  -> El {(1-comp.loc["Ledoit-Wolf","shrinkage"])*100:.1f}% conserva la covarianza muestral')


In [ ]:
# Visualizar distribucion de eigenvalores — muestral vs LW
cov_m = ret_clean.cov().values
cov_lw, alpha_lw = get_cov_matrix(ret_clean, metodo='ledoit_wolf')

eig_m  = np.sort(np.linalg.eigvalsh(cov_m))[::-1]
eig_lw = np.sort(np.linalg.eigvalsh(cov_lw))[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eigenvalores
ax = axes[0]
ax.semilogy(range(1, len(eig_m)+1), eig_m, 'o-', color='#e74c3c',
            linewidth=2, markersize=4, label='Muestral')
ax.semilogy(range(1, len(eig_lw)+1), eig_lw, 's-', color='#2ecc71',
            linewidth=2, markersize=4, label='Ledoit-Wolf')
ax.set_xlabel('Eigenvalor #')
ax.set_ylabel('Valor (escala log)')
ax.set_title('Espectro de eigenvalores\n(LW eleva los eigenvalores pequeños → estabilidad)', fontsize=11)
ax.legend()
ax.grid(alpha=0.15)
ax.fill_between(range(1, len(eig_m)+1), eig_m, eig_lw,
                where=eig_lw > eig_m, alpha=0.15, color='#2ecc71',
                label='Incremento LW')

# Heatmap de correlacion (LW)
ax2 = axes[1]
corr_lw = np.diag(1/np.sqrt(np.diag(cov_lw))) @ cov_lw @ np.diag(1/np.sqrt(np.diag(cov_lw)))
im = ax2.imshow(corr_lw, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax2.set_title(f'Matriz de correlacion Ledoit-Wolf\n(alpha={alpha_lw:.3f})', fontsize=11)
ax2.set_xlabel('Fondo')
ax2.set_ylabel('Fondo')
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.savefig('outputs/05_ledoit_wolf.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Clustering de Fondos <a id='5-clustering'></a>

La clasificación oficial (conservador/moderado/agresivo) puede no reflejar el **comportamiento real** de los fondos.

Aplicamos dos métodos complementarios:
1. **K-Means** sobre features estadísticos [μ, σ, skewness, max_drawdown, autocorr(1)], con k seleccionado por silhouette score
2. **Clustering Jerárquico** (Ward) usando distancia de Mantegna: $d(i,j) = \sqrt{0.5(1-\rho_{ij})}$



In [ ]:
# K-Means: seleccion automatica de k por silhouette
labels_km, k_opt, silhouettes, X_feat = kmeans_fondos(retornos)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Silhouette scores
ax = axes[0]
ks = list(silhouettes.keys())
vs = list(silhouettes.values())
bars = ax.bar(ks, vs, color='#3498db', edgecolor='white', linewidth=0.5)
ax.axvline(k_opt, color='#FFD700', linewidth=2, linestyle='--', label=f'k optimo = {k_opt}')
for bar, val in zip(bars, vs):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.005,
            f'{val:.3f}', ha='center', fontsize=9)
ax.set_xlabel('Numero de clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title(f'Seleccion de k por Silhouette\n(k={k_opt} seleccionado, score={silhouettes[k_opt]:.3f})', fontsize=11)
ax.legend()
ax.grid(axis='y', alpha=0.2)

# Scatter clusters en espacio retorno-volatilidad
ax2 = axes[1]
colores_cluster = plt.cm.tab10(np.linspace(0, 1, k_opt))
for c_id in range(1, k_opt+1):
    fondos_c = labels_km[labels_km == c_id].index
    ret_c = [X_feat.loc[f,'retorno_anual']*100 for f in fondos_c if f in X_feat.index]
    vol_c = [X_feat.loc[f,'volatilidad_anual']*100 for f in fondos_c if f in X_feat.index]
    nombres_c = [meta.loc[f,'nombre'][:15] if f in meta.index else f for f in fondos_c if f in X_feat.index]
    ax2.scatter(vol_c, ret_c, c=[colores_cluster[c_id-1]], s=100,
                label=f'Cluster {c_id}', zorder=5)
    for x, y, n in zip(vol_c, ret_c, nombres_c):
        ax2.annotate(n, (x, y), fontsize=6, xytext=(4,2), textcoords='offset points')
ax2.set_xlabel('Volatilidad anual (%)')
ax2.set_ylabel('Retorno anual (%)')
ax2.set_title('Clusters K-Means en espacio Riesgo-Retorno', fontsize=11)
ax2.legend(fontsize=7, ncol=2)
ax2.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/06_kmeans.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Clustering jerarquico — dendrograma
lm, corr, dist = clustering_jerarquico(retornos)
labels_hier = asignar_clusters_jerarquico(retornos, n_clusters=k_opt)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Dendrograma
ax = axes[0]
fondos_labels = [meta.loc[f,'nombre'].replace('Santander ','SAN ').replace('LarrainVial ','LV ')[:22]
                 if f in meta.index else f
                 for f in retornos.dropna(how='all',axis=1).columns]
hierarchy.dendrogram(lm, labels=fondos_labels, ax=ax,
                     leaf_rotation=90, leaf_font_size=7,
                     color_threshold=lm[-k_opt+1, 2],
                     above_threshold_color='white')
ax.set_title(f'Dendrograma Jerarquico (Ward)\n{k_opt} clusters — distancia Mantegna', fontsize=11)
ax.set_ylabel('Distancia')
ax.axhline(lm[-k_opt+1, 2], color='#FFD700', linewidth=1.5,
           linestyle='--', label=f'Corte en {k_opt} clusters')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.15)

# Comparacion clasificacion oficial vs cluster
ax2 = axes[1]
comp_df = meta.copy()
comp_df['cluster_km'] = labels_km
comp_df['cluster_hier'] = labels_hier

cross = pd.crosstab(comp_df['perfil'], comp_df['cluster_km'])
im = ax2.imshow(cross.values, cmap='Blues', aspect='auto')
ax2.set_xticks(range(len(cross.columns)))
ax2.set_xticklabels([f'C{c}' for c in cross.columns])
ax2.set_yticks(range(len(cross.index)))
ax2.set_yticklabels([p.capitalize() for p in cross.index])
for i in range(len(cross.index)):
    for j in range(len(cross.columns)):
        ax2.text(j, i, str(cross.values[i,j]), ha='center', va='center',
                 fontsize=11, color='white' if cross.values[i,j] > 3 else 'black')
ax2.set_title('Clasificacion CMF vs Clusters K-Means\n(fondos del mismo perfil oficial en distintos clusters)', fontsize=11)
ax2.set_xlabel('Cluster K-Means')
ax2.set_ylabel('Perfil CMF')
plt.colorbar(im, ax=ax2, label='N fondos')

plt.tight_layout()
plt.savefig('outputs/07_clustering_jerarquico.png', dpi=150, bbox_inches='tight')
plt.show()

# Resumen clusters
print('\nResumen por cluster (K-Means):')
resumen = resumen_clusters(labels_km, retornos, meta)
for _, r in resumen.iterrows():
    print(f'  Cluster {int(r.cluster)} | {r.n_fondos} fondos | '
          f'Ret: {r.retorno_anual:.1%} | Vol: {r.volatilidad_anual:.1%} | '
          f'Perfil dominante: {r.perfil_dominante} | Corredoras: {r.corredoras}')


## 6. Regímenes de Mercado — Hidden Markov Model <a id='6-hmm'></a>

El mercado alterna entre estados ocultos con distintas distribuciones de retorno.
El HMM Gaussiano aprende estos estados a partir de los datos históricos.

El número de regímenes se selecciona automáticamente minimizando el **BIC**:
$$BIC = -2 \cdot \log L + k \cdot \log(n)$$



In [ ]:
try:
    from src.regimes import (ajustar_hmm, seleccionar_n_regimenes,
                              retornos_por_regimen, probabilidad_transicion)
    modelo, estados, n_reg, params, bic_scores = ajustar_hmm(retornos, meta)
    HMM_OK = True
    print(f'HMM ajustado: {n_reg} regimenes')
    print(f'\nParametros por regimen:')
    for reg, p in params.items():
        print(f'  Regimen {reg} ({p["etiqueta"]:<8}): '
              f'Ret anual={p["media_anual"]:.1%} | '
              f'Vol anual={p["vol_anual"]:.1%} | '
              f'Frecuencia={p["frecuencia"]:.1%}')
except Exception as e:
    print(f'hmmlearn no disponible ({e}), usando fallback.')
    from src.regimes import _hmm_fallback
    _, estados, n_reg, params, bic_scores = _hmm_fallback(retornos, meta)
    HMM_OK = False


In [ ]:
# Visualizar regimenes en el tiempo
ret_mercado = retornos[meta[meta.perfil=='agresivo'].index].mean(axis=1).dropna()
estados_alineados = estados.reindex(ret_mercado.index).fillna(method='ffill')

COLORES_REG = {0: '#e74c3c', 1: '#f1c40f', 2: '#2ecc71', 3: '#3498db'}
ETIQUETAS_REG = {i: params[i]['etiqueta'] for i in params}

fig, axes = plt.subplots(3, 1, figsize=(16, 11),
                          gridspec_kw={'height_ratios': [2.5, 1, 1]})

# Retorno del mercado coloreado por regimen
ax = axes[0]
for reg in sorted(estados_alineados.unique()):
    mask = estados_alineados == reg
    fechas_reg = ret_mercado[mask].index
    if len(fechas_reg) > 0:
        ax.axvspan(fechas_reg[0], fechas_reg[-1],
                   alpha=0.12, color=COLORES_REG.get(reg, '#888'))

ret_cum = (1 + ret_mercado).cumprod() * 100
ax.plot(ret_cum.index, ret_cum.values, color='white', linewidth=2)
ax.set_ylabel('Retorno acumulado (base 100)')
ax.set_title(f'Retorno acumulado del mercado con regimenes HMM ({n_reg} estados)', fontsize=12)
for reg, p in params.items():
    ax.axhspan(0, 0, alpha=0.3, color=COLORES_REG.get(reg,'#888'),
               label=f'Reg {reg}: {p["etiqueta"]} (ret={p["media_anual"]:.1%})')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.15)

# Estado HMM en el tiempo
ax2 = axes[1]
ax2.step(estados_alineados.index, estados_alineados.values,
         where='post', color='#FFD700', linewidth=1.5)
ax2.set_yticks(list(range(n_reg)))
ax2.set_yticklabels([params[i]['etiqueta'] for i in range(n_reg)], fontsize=9)
ax2.set_ylabel('Regimen')
ax2.set_title('Estado del mercado en el tiempo', fontsize=11)
ax2.grid(alpha=0.15)

# Retornos mensuales del mercado
ax3 = axes[2]
colores_bar = [COLORES_REG.get(int(estados_alineados.get(f, 1)), '#888')
               for f in ret_mercado.index]
ax3.bar(ret_mercado.index, ret_mercado.values * 100,
        color=colores_bar, alpha=0.8, width=20)
ax3.axhline(0, color='white', linewidth=0.8)
ax3.set_ylabel('Retorno mensual (%)')
ax3.set_title('Retornos mensuales coloreados por regimen', fontsize=11)
ax3.grid(axis='y', alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/08_hmm_regimenes.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Retornos por regimen y perfil
from src.regimes import retornos_por_regimen
df_reg = retornos_por_regimen(retornos, estados_alineados, meta)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(n_reg)
width = 0.25
for i, perfil in enumerate(['conservador', 'moderado', 'agresivo']):
    sub = df_reg[df_reg.perfil == perfil].sort_values('regimen')
    if not sub.empty:
        bars = ax.bar(x[:len(sub)] + i*width, sub['ret_med']*100,
                      width, label=perfil.capitalize(), color=COLORS[perfil],
                      edgecolor='white', linewidth=0.5)

ax.set_xticks(x + width)
ax.set_xticklabels([params.get(i, {}).get('etiqueta', f'Reg {i}') for i in range(n_reg)])
ax.axhline(0, color='white', linewidth=1)
ax.set_ylabel('Retorno anual promedio (%)')
ax.set_title('Retorno anual por regimen y perfil\n(como se comporta cada tipo de fondo en cada estado del mercado)', fontsize=11)
ax.legend()
ax.grid(axis='y', alpha=0.15)
plt.tight_layout()
plt.savefig('outputs/09_retornos_por_regimen.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInterpretacion:')
for reg, p in params.items():
    print(f'  Regimen {reg} ({p["etiqueta"]}): {p["frecuencia"]:.0%} del tiempo | '
          f'ret={p["media_anual"]:.1%} | vol={p["vol_anual"]:.1%}')


## 7. Optimización de Portafolios <a id='7-optimizacion'></a>

Tres enfoques de optimización:
1. **Markowitz clásico con Ledoit-Wolf**: maximiza Sharpe con covarianza estabilizada
2. **Optimización robusta**: maximiza el Sharpe en el **peor escenario** posible (min-max)
3. **Óptimo global**: sin restricción de perfil, pesos 0-49%, mínimo 3 fondos



In [ ]:
print('Optimizando portafolios (4 perfiles + global con SP500)...')
portafolios = optimizar_todos(retornos, meta, robusto=True)

print('\n=== RESUMEN — 4 PERFILES DE INVERSION + OPTIMO GLOBAL ===')
for p_id, res in portafolios.items():
    if '_robusto' in p_id:
        continue
    sp_peso = res['composicion'].get('SP500', 0)
    sp_str  = f' | SP500 en cartera: {sp_peso:.1%}' if sp_peso > 0 else ''
    shrink  = f' | shrinkage={res["shrinkage"]:.3f}' if res.get('shrinkage') else ''
    print(f'\n{res["label"]}')
    print(f'  Sharpe: {res["sharpe"]:.3f} | Ret: {res["ret_anual"]:.2%} | '
          f'Vol: {res["vol_anual"]:.2%} | Activos: {res["n_activos"]}{sp_str}{shrink}')


In [ ]:
# Grafico comparativo: Markowitz vs Robusto vs Optimo + frontera eficiente
frontera = frontera_eficiente(retornos)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Frontera eficiente
ax = axes[0]
ax.plot(frontera['vol_anual']*100, frontera['ret_anual']*100,
        color='white', linewidth=2, label='Frontera eficiente (LW)', zorder=3)

# Fondos individuales
for perfil in ['conservador','moderado','agresivo']:
    sub = stats[stats['perfil'] == perfil]
    ax.scatter(sub['volatilidad_anual']*100, sub['retorno_anual']*100,
               c=COLORS[perfil], s=50, alpha=0.6, zorder=4)

# Portafolios
markers = {'conservador':'D','moderado':'D','agresivo':'D',
           'conservador_robusto':'v','moderado_robusto':'v','agresivo_robusto':'v',
           'optimo':'*'}
sizes   = {k: 300 if k=='optimo' else 150 for k in portafolios}
for p_id, res in portafolios.items():
    ax.scatter(res['vol_anual']*100, res['ret_anual']*100,
               c=res['color'], s=sizes[p_id],
               marker=markers.get(p_id,'o'), zorder=6,
               edgecolors='white', linewidth=1,
               label=res['label'])
    ax.annotate(res['label'][:15],
                (res['vol_anual']*100, res['ret_anual']*100),
                xytext=(8,4), textcoords='offset points',
                fontsize=7, color=res['color'])

ax.axhline(TPM_ANUAL*100, color='gray', linestyle=':', label=f'TPM {TPM_ANUAL:.1%}')
ax.set_xlabel('Volatilidad anual (%)')
ax.set_ylabel('Retorno anual (%)')
ax.set_title('Frontera Eficiente de Markowitz (Ledoit-Wolf)', fontsize=11)
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.15)

# Comparacion LW vs Robusto por perfil
ax2 = axes[1]
perfiles_base = ['conservador','moderado','agresivo']
x = np.arange(len(perfiles_base))
w = 0.35
sharpes_lw  = [portafolios[p]['sharpe'] for p in perfiles_base if p in portafolios]
sharpes_rob = [portafolios[p+'_robusto']['sharpe']
               for p in perfiles_base if p+'_robusto' in portafolios]

ax2.bar(x - w/2, sharpes_lw,  w, label='Markowitz LW',  color='#3498db', edgecolor='white')
ax2.bar(x + w/2, sharpes_rob, w, label='Robusto (e=0.1)',color='#e67e22', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels([p.capitalize() for p in perfiles_base])
ax2.set_ylabel('Sharpe Ratio')
ax2.set_title('Markowitz LW vs Optimizacion Robusta\n(robusto sacrifica algo de Sharpe por mayor estabilidad)', fontsize=11)
ax2.legend()
ax2.grid(axis='y', alpha=0.2)
ax2.axhline(0, color='white', linewidth=0.8)

plt.tight_layout()
plt.savefig('outputs/10_optimizacion.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Composicion de portafolios en tortas
n_port = len(portafolios)
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes_flat = axes.flatten()

for i, (p_id, res) in enumerate(portafolios.items()):
    if i >= len(axes_flat):
        break
    ax = axes_flat[i]
    labels_pie = [meta.loc[f,'nombre'].replace('Santander ','SAN ').replace('LarrainVial ','LV ')[:20]
                  if f in meta.index else f for f in res['composicion']]
    valores_pie = list(res['composicion'].values())
    wedges, texts, autotexts = ax.pie(
        valores_pie, labels=labels_pie, autopct='%1.0f%%',
        startangle=90, textprops={'fontsize': 7})
    for at in autotexts:
        at.set_fontsize(7)
    ax.set_title(f'{res["label"]}\nSharpe={res["sharpe"]:.3f} | Ret={res["ret_anual"]:.1%}',
                 color=res['color'], fontsize=9)

# Ocultar axes sobrantes
for i in range(len(portafolios), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle('Composicion de portafolios optimos', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/11_composicion_portafolios.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Walk-Forward Validation <a id='8-backtesting'></a>

El backtesting walk-forward evalúa si el portafolio óptimo **funciona en datos no vistos**.

```
|─── Train 1 (70%) ───|── Test 1 ──|
       |─── Train 2 (70%) ───|── Test 2 ──|
              |─── Train 3 (70%) ───|── Test 3 ──|
```

Si el Sharpe out-of-sample colapsa respecto al in-sample → **overfitting**.



In [ ]:
print('Ejecutando walk-forward validation (4 splits)...')
resultados_bt, df_bt = walk_forward_validation(
    retornos, meta, perfil='optimo', n_splits=4)

deg = degradacion_sharpe(df_bt)

print('\nResultados walk-forward:')
print(df_bt[['split','periodo_test','sharpe_insample','sharpe_outsample',
              'ret_anual_test','vol_anual_test','max_drawdown_test']].to_string(index=False))

print(f'\nAnalisis de degradacion IS → OOS:')
print(f'  Sharpe promedio in-sample  : {deg.get("sharpe_is_promedio",0):.3f}')
print(f'  Sharpe promedio out-sample : {deg.get("sharpe_oos_promedio",0):.3f}')
print(f'  Degradacion promedio       : {deg.get("degradacion_promedio",0):.3f}')
print(f'  Splits con Sharpe OOS > 0  : {deg.get("pct_splits_positivos",0):.0%}')
print()
if deg.get('degradacion_promedio', 0) > -0.2:
    print('CONCLUSION: Sin evidencia de overfitting significativo.')
else:
    print('ATENCION: Degradacion importante IS → OOS. Posible overfitting.')


In [ ]:
# Grafico walk-forward
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sharpe IS vs OOS por split
ax = axes[0]
splits = df_bt['split'].tolist()
sh_is  = df_bt['sharpe_insample'].tolist()
sh_oos = df_bt['sharpe_outsample'].tolist()
x = np.arange(len(splits))
w = 0.35
ax.bar(x - w/2, sh_is,  w, label='In-Sample',     color='#3498db', edgecolor='white')
ax.bar(x + w/2, sh_oos, w, label='Out-of-Sample', color='#e74c3c', edgecolor='white')
ax.axhline(0, color='white', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels([df_bt.loc[i,'periodo_test'][:7] for i in range(len(df_bt))], fontsize=8)
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe In-Sample vs Out-of-Sample\npor ventana de validacion', fontsize=11)
ax.legend()
ax.grid(axis='y', alpha=0.2)

# Retorno acumulado por split (OOS)
ax2 = axes[1]
colores_splits = plt.cm.coolwarm(np.linspace(0, 1, len(resultados_bt)))
for i, r in enumerate(resultados_bt):
    fechas = pd.date_range(
        r['fecha_test_inicio'], r['fecha_test_fin'], freq='MS')
    comp = r['composicion']
    fondos_disp = [f for f in comp if f in retornos.columns]
    if not fondos_disp:
        continue
    pesos_n = {f: comp[f]/sum(comp[f2] for f2 in fondos_disp) for f in fondos_disp}
    w_v = np.array([pesos_n[f] for f in fondos_disp])
    R_v = retornos.loc[r['fecha_test_inicio']:r['fecha_test_fin'], fondos_disp].dropna()
    if R_v.empty:
        continue
    ret_port = R_v.values @ w_v
    cum = (1 + ret_port).cumprod() * 100
    ax2.plot(R_v.index[:len(cum)], cum,
             color=colores_splits[i], linewidth=2,
             label=f'Split {r["split"]}: OOS Sharpe={r["sharpe_outsample"]:.2f}')

ax2.axhline(100, color='white', linewidth=1, linestyle='--')
ax2.set_ylabel('Retorno acumulado (base 100)')
ax2.set_title('Retorno acumulado OOS por split\n(portafolio optimo en datos no vistos)', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/12_walk_forward.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Bootstrap por Bloques vs Monte Carlo Normal <a id='9-bootstrap'></a>

El Monte Carlo Normal asume retornos i.i.d. Gaussianos — **ignorando autocorrelación y colas pesadas**.

El **Block Bootstrap** remuestrea bloques consecutivos del historial real:
```
Historial: [r₁, r₂, ..., r₇₅]
Bloque 1 (L=6): [r₁₂, r₁₃, r₁₄, r₁₅, r₁₆, r₁₇]
Bloque 2 (L=6): [r₄₅, r₄₆, r₄₇, r₄₈, r₄₉, r₅₀]
...
```
Esto **preserva la autocorrelación temporal** y los eventos extremos reales.



In [ ]:
# Proyeccion a 5 anos: Bootstrap vs Monte Carlo
opt = portafolios.get('optimo', list(portafolios.values())[0])
N_MESES = 60

print('Calculando simulaciones (1000 cada metodo)...')
bs = proyectar_bootstrap(retornos, opt['composicion'], MONTO, N_MESES,
                          n_sim=1000, block_size=6)
mc = proyectar_montecarlo_normal(retornos, opt['composicion'], MONTO, N_MESES,
                                  n_sim=1000)

df_comp, _, _ = comparar_metodos(retornos, opt['composicion'], MONTO, N_MESES)
print('\nComparacion Bootstrap vs Monte Carlo Normal:')
print(df_comp.to_string(index=False))

# Diferencia porcentual en P5 (cola pesimista)
p5_bs = bs['p5'][-1]
p5_mc = mc['p5'][-1]
print(f'\nDiferencia en P5 (cola pesimista): {(p5_bs-p5_mc)/p5_mc:.1%}')
print(f'  Bootstrap P5: ${p5_bs:,.0f}')
print(f'  MC Normal P5: ${p5_mc:,.0f}')
print(f'Bootstrap es mas conservador en la cola pesimista.')


In [ ]:
# Grafico comparativo Bootstrap vs MC Normal
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Trayectorias medianas con bandas
ax = axes[0]
for metodo, proy, color, label in [
    (bs, bs, '#2ecc71', 'Bootstrap Bloques'),
    (mc, mc, '#e74c3c', 'Monte Carlo Normal')
]:
    f = proy['fechas']
    ax.fill_between(f, proy['p5'], proy['p95'],
                    alpha=0.10, color=color, label=f'{label} 5-95%')
    ax.fill_between(f, proy['p25'], proy['p75'],
                    alpha=0.20, color=color)
    ax.plot(f, proy['p50'], color=color, linewidth=2.5,
            label=f'{label} mediana (${proy["mediana"]/1e6:.1f}M)')

ax.axhline(MONTO, color='white', linewidth=1, linestyle='--', label='Inversion inicial')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.set_title('Proyeccion 5 anos: Bootstrap vs Monte Carlo\n(portafolio optimo global)', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.15)

# Distribucion de valores finales
ax2 = axes[1]
finales_bs = bs['simulaciones'][:,-1]
finales_mc = mc['simulaciones'][:,-1]
bins = np.linspace(min(finales_bs.min(), finales_mc.min()),
                   max(finales_bs.max(), finales_mc.max()), 60)
ax2.hist(finales_mc/1e6, bins=bins/1e6, alpha=0.5, color='#e74c3c',
         density=True, label='Monte Carlo Normal', edgecolor='white', linewidth=0.2)
ax2.hist(finales_bs/1e6, bins=bins/1e6, alpha=0.5, color='#2ecc71',
         density=True, label='Bootstrap Bloques', edgecolor='white', linewidth=0.2)
ax2.axvline(MONTO/1e6, color='white', linewidth=1.5, linestyle='--', label='Inicial')
ax2.axvline(np.median(finales_bs)/1e6, color='#2ecc71', linewidth=2, linestyle='-')
ax2.axvline(np.median(finales_mc)/1e6, color='#e74c3c', linewidth=2, linestyle='-')
ax2.set_xlabel('Valor final ($M CLP)')
ax2.set_ylabel('Densidad')
ax2.set_title('Distribucion de valores finales (5 anos)\nBootstrap tiene colas mas anchas y realistas', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/13_bootstrap_vs_mc.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Resumen Ejecutivo <a id='10-resumen'></a>


In [ ]:
# Simulacion historica — 5 escenarios (4 perfiles + optimo)
port_sel = {k: portafolios[k]
            for k in ['conservador','moderado','agresivo','sp500','optimo']
            if k in portafolios}

fig, (ax, ax_dd) = plt.subplots(2, 1, figsize=(16, 10),
                                  gridspec_kw={'height_ratios': [3, 1]})
sims = {}
for p_id, res in port_sel.items():
    sim = simular_historico(retornos, res['composicion'], MONTO)
    sims[p_id] = sim
    lw = 3.0 if p_id in ['optimo','sp500'] else 2.0
    ls = '-.' if p_id == 'sp500' else '-'
    ax.plot(sim.index, sim.values, color=res['color'], linewidth=lw,
            linestyle=ls,
            label=f'{res["label"]} (${sim.iloc[-1]/1e6:.2f}M)',
            zorder=6 if p_id in ['optimo','sp500'] else 4)
    dd = drawdown_serie(sim)
    ax_dd.fill_between(dd.index, dd.values, 0, color=res['color'], alpha=0.25)
    ax_dd.plot(dd.index, dd.values, color=res['color'], linewidth=1)

ax.axhline(MONTO, color='gray', linewidth=1, linestyle='--', label='Inversion inicial')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.set_title(f'Crecimiento historico de ${MONTO/1e6:.0f}M CLP — 5 escenarios (2020-2026)\n'
             f'SP500 en linea punteada para comparacion', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.15)
ax_dd.set_ylabel('Drawdown (%)')
ax_dd.set_title('Drawdown desde el maximo historico', fontsize=11)
ax_dd.grid(alpha=0.15)
plt.tight_layout()
plt.savefig('outputs/14_simulacion_historica.png', dpi=150, bbox_inches='tight')
plt.show()

# Tabla resumen
rows = []
for p_id, sim in sims.items():
    res = port_sel[p_id]
    rows.append({
        'Perfil':        res['label'],
        'Inversion':     f'${MONTO:,.0f}',
        'Valor final':   f'${sim.iloc[-1]:,.0f}',
        'Retorno total': f'{(sim.iloc[-1]/MONTO-1):.1%}',
        'Maximo':        f'${sim.max():,.0f}',
        'Drawdown max':  f'{drawdown_serie(sim).min():.1%}',
    })
pd.DataFrame(rows)


In [ ]:
# Tabla resumen final
print('='*75)
print('RESUMEN EJECUTIVO — OPTIMIZACION FONDOS MUTUOS CHILE')
print('='*75)
print(f'Universo: 42 fondos | Santander (20) + LarrainVial (22)')
print(f'Periodo : 2020-2026 | {len(retornos)} meses | TPM ref: {TPM_ANUAL:.1%}')
print()

# Portafolios
print('PORTAFOLIOS OPTIMOS (Markowitz + Ledoit-Wolf)')
print('-'*75)
print(f'{"Portafolio":<22} {"Sharpe":>7} {"Ret.Anual":>10} {"Vol.Anual":>10} '
      f'{"MaxDD":>8} {"Valor Final":>14}')
print('-'*75)
for p_id, res in port_sel.items():
    sim = sims[p_id]
    from src.metrics import max_drawdown
    r_port = retornos[[f for f in res['composicion'] if f in retornos.columns]].dropna()
    w = np.array([res['composicion'][f] for f in r_port.columns
                  if f in res['composicion']])
    if len(w) > 0:
        ret_port = r_port.values @ w / w.sum()
        mdd = max_drawdown(ret_port)
    else:
        mdd = 0
    star = ' ★' if p_id == 'optimo' else ''
    print(f'{res["label"]+star:<22} {res["sharpe"]:>7.3f} {res["ret_anual"]:>10.2%} '
          f'{res["vol_anual"]:>10.2%} {mdd:>8.1%} ${sim.iloc[-1]/1e6:>10.2f}M')

print()
print('PROYECCION MONTE CARLO BOOTSTRAP — 5 ANOS (portafolio optimo)')
print('-'*75)
print(f'  Minimo  (peor caso)  : ${bs["minimo"]/1e6:.2f}M')
print(f'  P25                  : ${bs["p25"][-1]/1e6:.2f}M')
print(f'  Mediana              : ${bs["mediana"]/1e6:.2f}M')
print(f'  P75                  : ${bs["p75"][-1]/1e6:.2f}M')
print(f'  Maximo (mejor caso)  : ${bs["maximo"]/1e6:.2f}M')

print()
print('HALLAZGOS CLAVE')
print('-'*75)
print(f'  1. Ledoit-Wolf reduce condition# de 4e8 a 130 (alpha=0.169)')
print(f'  2. K-Means identifica {k_opt} clusters que cruzan la clasificacion CMF oficial')
print(f'  3. Walk-forward: degradacion IS→OOS = {deg.get("degradacion_promedio",0):.3f} (sin overfitting)')
print(f'  4. Bootstrap mas conservador que MC Normal en cola pesimista (P5)')
print(f'  5. Sharpe optimo: {portafolios["optimo"]["sharpe"]:.3f} (vs promedio industria ~0.2)')
print()
print('Disclaimer: Este analisis es educativo. No constituye asesoria financiera.')


## 11. SP500 como Benchmark y Componente del Portafolio <a id='11-sp500'></a>

El SP500 en CLP es el benchmark global de referencia. Lo analizamos como:
1. **Benchmark independiente** — comparación de retorno y riesgo
2. **Componente del portafolio óptimo** — con peso máximo 40%



In [ ]:
from src.benchmarks import (cargar_sp500, get_retornos_sp500, construir_proxy_ipsa,
                            tabla_alpha, stress_test, atribucion_retorno)

# Cargar SP500 y IPSA proxy
sp500_df    = cargar_sp500('data/raw/sp500.csv')
ret_sp500   = get_retornos_sp500('data/raw/sp500.csv')
ret_ipsa, fondos_ipsa = construir_proxy_ipsa(retornos, meta)

print('SP500 (en CLP):')
print(f'  Retorno anual : {ret_sp500.mean()*12:.2%}')
print(f'  Volatilidad   : {ret_sp500.std()*12**0.5:.2%}')
print(f'  Sharpe        : {(ret_sp500.mean()*12 - TPM_ANUAL)/(ret_sp500.std()*12**0.5):.3f}')
print()
print('IPSA proxy (fondos acciones Chile):')
print(f'  Fondos usados : {len(fondos_ipsa)}')
print(f'  Retorno anual : {ret_ipsa.mean()*12:.2%}')
print(f'  Volatilidad   : {ret_ipsa.std()*12**0.5:.2%}')
print(f'  Sharpe        : {(ret_ipsa.mean()*12 - TPM_ANUAL)/(ret_ipsa.std()*12**0.5):.3f}')


In [ ]:
# Retorno acumulado: portafolios vs SP500 vs IPSA
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
# Portafolios
port_sel = {k: portafolios[k] for k in ['conservador','moderado','agresivo','optimo']
            if k in portafolios}
for p_id, res in port_sel.items():
    sim = simular_historico(retornos, res['composicion'], MONTO)
    lw  = 3.0 if p_id == 'optimo' else 1.8
    ax.plot(sim.index, sim.values, color=res['color'], linewidth=lw,
            label=f'{res["label"]} ({(sim.iloc[-1]/MONTO-1)*100:.0f}%)', zorder=5)

# SP500 en CLP
sp_align = ret_sp500.reindex(retornos.index).dropna()
sp_cum = (1 + sp_align).cumprod() * MONTO
ax.plot(sp_cum.index, sp_cum.values, color='#00b4d8', linewidth=2.5,
        linestyle='-.', label=f'SP500 CLP ({(sp_cum.iloc[-1]/MONTO-1)*100:.0f}%)', zorder=6)

# IPSA proxy
ip_align = ret_ipsa.reindex(retornos.index).dropna()
ip_cum = (1 + ip_align).cumprod() * MONTO
ax.plot(ip_cum.index, ip_cum.values, color='#ff9f1c', linewidth=2.0,
        linestyle=':', label=f'IPSA proxy ({(ip_cum.iloc[-1]/MONTO-1)*100:.0f}%)', zorder=5)

ax.axhline(MONTO, color='gray', linewidth=1, linestyle='--')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M'))
ax.set_title('Retorno acumulado: Portafolios vs SP500 vs IPSA\n(2020-2026, $10M inicial)', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.15)

# Riesgo vs retorno incluyendo benchmarks
ax2 = axes[1]
for perfil in ['conservador','moderado','agresivo']:
    sub = stats[stats.perfil == perfil]
    ax2.scatter(sub.volatilidad_anual*100, sub.retorno_anual*100,
                c=COLORS[perfil], s=40, alpha=0.5, zorder=3)

for p_id, res in port_sel.items():
    ax2.scatter(res['vol_anual']*100, res['ret_anual']*100,
                c=res['color'], s=200, marker='*', zorder=6,
                edgecolors='white', linewidth=1, label=res['label'])

# Benchmarks
sp_ret = ret_sp500.mean()*12
sp_vol = ret_sp500.std()*12**0.5
ip_ret = ret_ipsa.mean()*12
ip_vol = ret_ipsa.std()*12**0.5

ax2.scatter(sp_vol*100, sp_ret*100, c='#00b4d8', s=250, marker='P',
            zorder=7, edgecolors='white', linewidth=1, label='SP500 (CLP)')
ax2.scatter(ip_vol*100, ip_ret*100, c='#ff9f1c', s=250, marker='P',
            zorder=7, edgecolors='white', linewidth=1, label='IPSA (proxy)')
ax2.annotate('SP500', (sp_vol*100, sp_ret*100), xytext=(6,4),
             textcoords='offset points', fontsize=9, color='#00b4d8')
ax2.annotate('IPSA', (ip_vol*100, ip_ret*100), xytext=(6,4),
             textcoords='offset points', fontsize=9, color='#ff9f1c')

ax2.axhline(TPM_ANUAL*100, color='gray', linestyle=':', label=f'TPM {TPM_ANUAL:.1%}')
ax2.set_xlabel('Volatilidad anual (%)')
ax2.set_ylabel('Retorno anual (%)')
ax2.set_title('Mapa Riesgo-Retorno con Benchmarks', fontsize=11)
ax2.legend(fontsize=8, ncol=2)
ax2.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/15_benchmarks_comparacion.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Portafolio optimo con SP500 incluido (peso max 40%)
from src.optimizer import optimizar_global
from src.metrics import simular_historico

# Agregar SP500 como activo adicional en la matriz de retornos
ret_sp500_alineado = ret_sp500.rename('SP500')
retornos_con_sp = retornos.copy()
retornos_con_sp['SP500'] = ret_sp500_alineado

# Metadata para SP500
import pandas as pd
meta_con_sp = meta.copy()
meta_con_sp.loc['SP500'] = {
    'nombre': 'S&P 500 (CLP)', 'perfil': 'agresivo',
    'corredora': 'externo', 'moneda_orig': 'USD'
}

print('Optimizando portafolio global CON SP500...')
opt_con_sp = optimizar_global(retornos_con_sp, meta_con_sp,
                               peso_min=0.0, peso_max=0.40)

print(f'\nOptimo SIN SP500: Sharpe={portafolios["optimo"]["sharpe"]:.3f} | '
      f'Ret={portafolios["optimo"]["ret_anual"]:.2%} | Vol={portafolios["optimo"]["vol_anual"]:.2%}')
print(f'Optimo CON SP500: Sharpe={opt_con_sp["sharpe"]:.3f} | '
      f'Ret={opt_con_sp["ret_anual"]:.2%} | Vol={opt_con_sp["vol_anual"]:.2%}')
print(f'\nComposicion optimo con SP500:')
for fid, peso in sorted(opt_con_sp['composicion'].items(), key=lambda x: -x[1]):
    nombre = meta_con_sp.loc[fid,'nombre'] if fid in meta_con_sp.index else fid
    print(f'  {peso:>5.1%}  {nombre}')


## 12. Tabla de Alpha vs Benchmarks <a id='12-alpha'></a>

El **Alpha de Jensen** mide el retorno en exceso sobre lo que predice el CAPM:

$$\alpha = E[r_p] - [r_f + \beta \cdot (E[r_m] - r_f)]$$

Un $\alpha > 0$ indica que el gestor genera valor por encima del riesgo sistémico asumido.



In [ ]:
df_alpha = tabla_alpha(portafolios, retornos, meta)

# Tabla formateada
df_alpha_show = df_alpha.copy()
for c in ['ret_anual','alpha_vs_sp500','alpha_vs_ipsa','tracking_err_sp500']:
    df_alpha_show[c] = df_alpha_show[c].map('{:.2%}'.format)
for c in ['sharpe','beta_vs_sp500','beta_vs_ipsa','r2_vs_sp500','ir_vs_sp500']:
    if c in df_alpha_show.columns:
        df_alpha_show[c] = df_alpha_show[c].map('{:.3f}'.format)
df_alpha_show.rename(columns={
    'portafolio':'Portafolio','ret_anual':'Ret. Anual','sharpe':'Sharpe',
    'alpha_vs_sp500':'Alpha vs SP500','beta_vs_sp500':'Beta vs SP500',
    'r2_vs_sp500':'R² vs SP500','ir_vs_sp500':'Info. Ratio vs SP500',
    'alpha_vs_ipsa':'Alpha vs IPSA','tracking_err_sp500':'Tracking Error',
}, inplace=True)
display(df_alpha_show)

print('\nInterpretacion:')
print('  Alpha > 0  → portafolio supera al benchmark ajustado por riesgo')
print('  Beta < 1   → menor exposicion sistémica que el benchmark')
print('  IR > 0.5   → retorno activo significativo por unidad de riesgo activo')


In [ ]:
# Grafico alpha vs benchmarks
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

portafolios_nombres = df_alpha['portafolio'].tolist()
x = np.arange(len(portafolios_nombres))
w = 0.35

ax = axes[0]
colores_alpha = ['#2ecc71' if v > 0 else '#e74c3c' for v in df_alpha['alpha_vs_sp500']]
bars1 = ax.bar(x - w/2, df_alpha['alpha_vs_sp500']*100, w,
               color=colores_alpha, edgecolor='white', label='Alpha vs SP500')
colores_ipsa = ['#2ecc71' if v > 0 else '#e74c3c' for v in df_alpha['alpha_vs_ipsa']]
bars2 = ax.bar(x + w/2, df_alpha['alpha_vs_ipsa']*100, w,
               color=colores_ipsa, edgecolor='white', alpha=0.7, label='Alpha vs IPSA')
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x()+bar.get_width()/2, h+0.1 if h>=0 else h-0.3,
            f'{h:.1f}%', ha='center', fontsize=8)
ax.axhline(0, color='white', linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(portafolios_nombres, fontsize=9, rotation=10)
ax.set_ylabel('Alpha anualizado (%)')
ax.set_title('Alpha de Jensen por portafolio\n(verde = supera al benchmark ajustado por riesgo)', fontsize=11)
ax.legend()
ax.grid(axis='y', alpha=0.2)

# Beta por portafolio
ax2 = axes[1]
ax2.bar(x - w/2, df_alpha['beta_vs_sp500'], w,
        color='#00b4d8', edgecolor='white', label='Beta vs SP500')
ax2.bar(x + w/2, df_alpha['beta_vs_ipsa'], w,
        color='#ff9f1c', edgecolor='white', alpha=0.7, label='Beta vs IPSA')
ax2.axhline(1.0, color='white', linewidth=1, linestyle='--', label='Beta = 1 (mercado)')
ax2.set_xticks(x)
ax2.set_xticklabels(portafolios_nombres, fontsize=9, rotation=10)
ax2.set_ylabel('Beta')
ax2.set_title('Beta por portafolio\n(< 1 = menor riesgo sistémico que el mercado)', fontsize=11)
ax2.legend()
ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('outputs/16_alpha_benchmarks.png', dpi=150, bbox_inches='tight')
plt.show()


## 13. Stress Testing — Escenarios Históricos <a id='13-stress'></a>

Evaluamos el comportamiento de cada portafolio en los períodos más relevantes de 2020-2026:
COVID crash, rally post-COVID, subida de TPM, crisis inflacionaria y normalización.



In [ ]:
df_stress = stress_test(portafolios, retornos, meta)

# Tabla formateada
df_stress_show = df_stress.copy()
cols_num = [c for c in df_stress_show.columns if c not in ['Escenario','Descripcion']]
for c in cols_num:
    df_stress_show[c] = df_stress_show[c].map(lambda x: f'{x:.1%}' if x is not None else 'N/D')
display(df_stress_show.drop(columns='Descripcion'))


In [ ]:
# Heatmap stress test
df_heat = df_stress.copy()
cols_num = [c for c in df_heat.columns if c not in ['Escenario','Descripcion']]
data_heat = df_heat[cols_num].values.astype(float)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(data_heat.T, cmap='RdYlGn', aspect='auto',
               vmin=-0.20, vmax=0.30)
ax.set_xticks(range(len(df_heat)))
ax.set_xticklabels(df_heat['Escenario'], fontsize=8, rotation=15, ha='right')
ax.set_yticks(range(len(cols_num)))
ax.set_yticklabels(cols_num, fontsize=9)
plt.colorbar(im, ax=ax, label='Retorno acumulado en el periodo')
ax.set_title('Stress Test — Retorno acumulado por escenario historico\n(verde=positivo, rojo=negativo)', fontsize=12)

for i in range(len(cols_num)):
    for j in range(len(df_heat)):
        val = data_heat[j, i]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                    fontsize=9, color='black' if abs(val) < 0.15 else 'white',
                    fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/17_stress_test.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nHallazgos clave:')
print('  COVID Crash: Conservador mas resistente (-1.2% vs -15.4% agresivo)')
print('  Crisis 2022: Conservador positivo (+11.5%) — tasas altas benefician renta fija')
print('  2023-2024  : Todos los perfiles positivos, SP500 lidera en la recuperacion')


## 14. Atribución de Retorno <a id='14-atribucion'></a>

El análisis de atribución descompone el retorno del portafolio en la **contribución individual de cada fondo**,
corredora y perfil. Permite identificar qué activos generan valor y cuáles son prescindibles.

**Contribución = Peso × Retorno anual del fondo**



In [ ]:
opt = portafolios['optimo']
df_fondo, df_corr, df_perfil = atribucion_retorno(opt, retornos, meta)

print(f'Retorno total atribuido: {df_fondo.contribucion.sum():.2%}')
print(f'  (vs retorno del portafolio: {opt["ret_anual"]:.2%} — diferencia por alineacion de fechas)')
print()
print('Contribucion por fondo:')
df_fondo_show = df_fondo[['nombre','corredora','perfil','peso','ret_anual','contribucion']].copy()
df_fondo_show['peso']         = df_fondo_show['peso'].map('{:.1%}'.format)
df_fondo_show['ret_anual']    = df_fondo_show['ret_anual'].map('{:.2%}'.format)
df_fondo_show['contribucion'] = df_fondo_show['contribucion'].map('{:.2%}'.format)
display(df_fondo_show)


In [ ]:
# Graficos de atribucion
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# Por fondo — waterfall
ax = axes[0]
nombres_cortos = [n.replace('Santander ','SAN ').replace('LarrainVial ','LV ')[:22]
                  for n in df_fondo['nombre']]
colores_contrib = ['#2ecc71' if v > 0 else '#e74c3c' for v in df_fondo['contribucion']]
bars = ax.barh(nombres_cortos, df_fondo['contribucion']*100,
               color=colores_contrib, edgecolor='white', linewidth=0.3)
for bar, val in zip(bars, df_fondo['contribucion']*100):
    ax.text(val + 0.02, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)
ax.axvline(0, color='white', linewidth=1)
ax.set_xlabel('Contribucion al retorno anual (%)')
ax.set_title('Atribucion por fondo\n(portafolio optimo global)', fontsize=11)
ax.grid(axis='x', alpha=0.15)

# Por corredora — torta
ax2 = axes[1]
colores_corr = {'santander': '#e74c3c', 'larrain_vial': '#3498db', 'externo': '#FFD700'}
wedges, texts, autotexts = ax2.pie(
    df_corr['contribucion_total'].abs(),
    labels=[f'{c}\n({v:.1%})' for c, v in zip(df_corr.corredora, df_corr.contribucion_total)],
    colors=[colores_corr.get(c,'#888') for c in df_corr.corredora],
    autopct='%1.0f%%', startangle=90, textprops={'fontsize':9})
for at in autotexts:
    at.set_fontsize(8)
ax2.set_title('Atribucion por corredora\n(% del retorno total)', fontsize=11)

# Por perfil — barras
ax3 = axes[2]
colores_p = [COLORS.get(p,'#888') for p in df_perfil.perfil]
bars3 = ax3.bar(df_perfil.perfil, df_perfil.contribucion_total*100,
                color=colores_p, edgecolor='white')
for bar, val in zip(bars3, df_perfil.contribucion_total*100):
    ax3.text(bar.get_x()+bar.get_width()/2, val+0.1,
             f'{val:.1f}%', ha='center', fontsize=9)
ax3.set_ylabel('Contribucion al retorno (%)')
ax3.set_title('Atribucion por perfil\n(que tipo de fondo genera mas retorno)', fontsize=11)
ax3.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('outputs/18_atribucion_retorno.png', dpi=150, bbox_inches='tight')
plt.show()


## 15. Hierarchical Risk Parity (HRP) <a id='15-hrp'></a>

López de Prado (2016) — *Journal of Portfolio Management*

HRP construye el portafolio en 3 pasos sin invertir la covarianza:
1. **Clustering jerárquico** → agrupa activos similares
2. **Quasi-diagonalización** → reordena la covarianza en bloques
3. **Bisección recursiva** → asigna capital inversamente proporcional a la varianza de cada cluster

**Diferencia clave:** Markowitz busca el máximo Sharpe (concentrado).
HRP distribuye el **riesgo** equitativamente (diversificado).



In [ ]:
from src.hrp import hrp_portfolio, comparar_hrp_markowitz
from scipy.cluster import hierarchy

print('Construyendo portafolio HRP...')
hrp = hrp_portfolio(retornos, meta)
print(f'HRP: Sharpe={hrp["sharpe"]:.3f} | Ret={hrp["ret_anual"]:.2%} | '
      f'Vol={hrp["vol_anual"]:.2%} | Activos={hrp["n_activos"]}')

comp_df = comparar_hrp_markowitz(hrp, portafolios.get('optimo'), retornos)
print('\nComparacion HRP vs Markowitz:')
comp_df['Concentracion HHI'] = comp_df['Concentracion'].map('{:.3f}'.format)
comp_df['Retorno anual']     = comp_df['Retorno anual'].map('{:.2%}'.format)
comp_df['Volatilidad']       = comp_df['Volatilidad'].map('{:.2%}'.format)
comp_df['Sharpe']            = comp_df['Sharpe'].map('{:.3f}'.format)
display(comp_df[['Metodo','Sharpe','Retorno anual','Volatilidad','N activos','Concentracion HHI']])
print('\nHHI (Herfindahl): 1/N=diversificado, 1=concentrado en un activo')


In [ ]:
# Dendrograma HRP + comparacion de pesos
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Dendrograma con el orden HRP
ax = axes[0]
lm = hrp['linkage']
sort_ix = hrp['sort_ix']
labels_d = [meta.loc[f,'nombre'].replace('Santander ','SAN ').replace('LarrainVial ','LV ')
            .replace('S&P 500','SP500')[:20]
            if f in meta.index else f for f in retornos.dropna(how='all',axis=1).columns]

hierarchy.dendrogram(lm, labels=labels_d, ax=ax,
                     leaf_rotation=90, leaf_font_size=7,
                     color_threshold=lm[-4,2],
                     above_threshold_color='white')
ax.set_title('Dendrograma HRP — estructura jerarquica de correlaciones\n'
             '(fondos del mismo cluster tienen comportamiento similar)', fontsize=11)
ax.set_ylabel('Distancia Mantegna')
ax.grid(axis='y', alpha=0.15)

# Comparacion pesos HRP vs Markowitz optimo
ax2 = axes[1]
opt = portafolios.get('optimo', {})
fondos_union = sorted(set(list(hrp['composicion'].keys()) +
                          list(opt.get('composicion',{}).keys())))
fondos_union = [f for f in fondos_union if f in retornos.columns][:20]  # top 20

nombres = [meta.loc[f,'nombre'].replace('Santander ','SAN ').replace('LarrainVial ','LV ')
           .replace('S&P 500 (CLP)','SP500')[:22]
           if f in meta.index else f for f in fondos_union]

pesos_hrp = [hrp['composicion'].get(f, 0) for f in fondos_union]
pesos_mkt = [opt.get('composicion',{}).get(f, 0) for f in fondos_union]

x = np.arange(len(fondos_union))
w = 0.4
ax2.barh([n + ' (HRP)' for n in nombres], pesos_hrp,
         color='#9b59b6', alpha=0.85, edgecolor='white', linewidth=0.3, label='HRP')
ax2.barh([n + ' (MKT)' for n in nombres], [-p for p in pesos_mkt],
         color='#FFD700', alpha=0.85, edgecolor='white', linewidth=0.3, label='Markowitz')
ax2.axvline(0, color='white', linewidth=1)
ax2.set_xlabel('← Markowitz    HRP →')
ax2.set_title('Distribucion de pesos\nHRP (morado) vs Markowitz LW (dorado)', fontsize=11)
ax2.legend()
ax2.grid(axis='x', alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/19_hrp.png', dpi=150, bbox_inches='tight')
plt.show()
print('Conclusion: HRP distribuye mas equitativamente, Markowitz concentra en los mejores activos.')


## 16. GARCH — Volatilidad Condicional <a id='16-garch'></a>

Bollerslev (1986) — modelo estándar para volatilidad variable en el tiempo.

$$\sigma_t^2 = \omega + \alpha \cdot \epsilon_{t-1}^2 + \beta \cdot \sigma_{t-1}^2$$

- **alpha** (ARCH): reacción a shocks → qué tan rápido sube la volatilidad
- **beta** (GARCH): persistencia → cuánto dura el período de alta volatilidad
- **alpha + beta**: persistencia total (< 1 para estacionariedad)

En 2022, la volatilidad de los fondos chilenos fue 3-4x mayor que en 2024.



In [ ]:
from src.garch import garch_todos_perfiles, vol_condicional_portafolio, var_garch

print('Ajustando modelos GARCH por perfil...')
res_garch = garch_todos_perfiles(retornos, meta)

# Tabla de parametros
rows_g = []
for perfil, res in res_garch.items():
    label = {'conservador':'Conservador','moderado':'Moderado',
             'agresivo':'Agresivo','sp500':'S&P 500'}.get(perfil, perfil)
    rows_g.append({
        'Perfil':      label,
        'Alpha (ARCH)':  f'{res["alpha"]:.3f}',
        'Beta (GARCH)':  f'{res["beta"]:.3f}',
        'Persistencia':  f'{res["persistencia"]:.3f}',
        'Vol incond.':   f'{res["vol_incondicional"]:.2%}',
    })
pd.DataFrame(rows_g)


In [ ]:
# Graficar volatilidad condicional vs constante por perfil
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes_flat = axes.flatten()

for i, (perfil, res) in enumerate(res_garch.items()):
    if i >= 4:
        break
    ax = axes_flat[i]
    color = {'conservador':'#2ecc71','moderado':'#f1c40f',
             'agresivo':'#e74c3c','sp500':'#00b4d8'}.get(perfil,'#888')
    label = {'conservador':'Conservador','moderado':'Moderado',
             'agresivo':'Agresivo','sp500':'S&P 500'}.get(perfil, perfil)

    vol_cond = res['vol_condicional']
    vol_hist = vol_cond.mean()  # volatilidad promedio (constante)

    ax.plot(vol_cond.index, vol_cond.values * 100,
            color=color, linewidth=2, label='Vol. GARCH (condicional)')
    ax.axhline(vol_hist * 100, color='white', linewidth=1.5,
               linestyle='--', label=f'Vol. historica: {vol_hist:.1%}')

    # Sombrear periodos criticos
    ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-05-01'),
               alpha=0.15, color='red', label='COVID crash')
    ax.axvspan(pd.Timestamp('2022-01-01'), pd.Timestamp('2023-06-01'),
               alpha=0.10, color='orange', label='Crisis inflacion')

    ax.set_title(f'{label}\nalpha={res["alpha"]:.3f} beta={res["beta"]:.3f} '
                 f'persist={res["persistencia"]:.3f}', color=color, fontsize=11)
    ax.set_ylabel('Volatilidad anual (%)')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.15)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))

fig.suptitle('Volatilidad condicional GARCH(1,1) por perfil\n'
             '(vs. volatilidad historica constante — linea punteada)', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/20_garch_volatilidad.png', dpi=150, bbox_inches='tight')
plt.show()
print('La volatilidad NO es constante — COVID y 2022 fueron periodos de riesgo excepcional.')


## 17. Correlación Dinámica y Métricas Rodantes <a id='17-rolling'></a>

Las correlaciones entre activos cambian en el tiempo. En crisis,
todos los activos tienden a correlacionarse hacia 1 (flight to quality),
reduciendo la diversificación exactamente cuando más se necesita.

**Rolling window = 12 meses** para capturar ciclos anuales.



In [ ]:
from src.rolling import (correlacion_media_rodante, volatilidad_rodante,
                         sharpe_rodante, beta_rodante, resumen_rolling)

# Correlacion media rodante
corr_media = correlacion_media_rodante(retornos, meta, ventana=12)

# Volatilidad rodante por perfil
vol_rolling = volatilidad_rodante(retornos, meta, ventana=12)

fig, axes = plt.subplots(3, 1, figsize=(16, 13))

# Panel 1: Correlacion media del sistema
ax = axes[0]
ax.plot(corr_media.index, corr_media.values,
        color='#FFD700', linewidth=2, label='Correlacion media (12m)')
ax.fill_between(corr_media.index, corr_media.values,
                corr_media.mean(), alpha=0.2, color='#FFD700')
ax.axhline(corr_media.mean(), color='white', linewidth=1,
           linestyle='--', label=f'Media historica: {corr_media.mean():.3f}')
ax.axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-05-01'),
           alpha=0.2, color='red', label='COVID')
ax.axvspan(pd.Timestamp('2022-01-01'), pd.Timestamp('2023-06-01'),
           alpha=0.1, color='orange', label='Crisis inflacion')
ax.set_ylabel('Correlacion media')
ax.set_title('Correlacion media del sistema (ventana 12 meses)\n'
             'Alta correlacion = menor diversificacion real', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.15)
ax.set_ylim(0, 1)

# Panel 2: Volatilidad rodante por perfil
ax2 = axes[1]
for col in vol_rolling.columns:
    color = {'conservador':'#2ecc71','moderado':'#f1c40f',
             'agresivo':'#e74c3c','sp500':'#00b4d8'}.get(col,'#888')
    ax2.plot(vol_rolling.index, vol_rolling[col]*100,
             color=color, linewidth=1.8, label=col.capitalize())
ax2.set_ylabel('Volatilidad anual (%)')
ax2.set_title('Volatilidad rodante por perfil (12 meses)\n'
              'Picos = periodos de mayor riesgo', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.15)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.0f}%'))

# Panel 3: Sharpe rodante del optimo global
ax3 = axes[2]
opt = portafolios.get('optimo',{})
if opt:
    sh_roll = sharpe_rodante(retornos, opt['composicion'], ventana=24)
    ax3.plot(sh_roll.index, sh_roll.values, color='#FFD700', linewidth=2)
    ax3.fill_between(sh_roll.index, sh_roll.values, 0,
                     where=sh_roll.values > 0, alpha=0.2, color='#2ecc71')
    ax3.fill_between(sh_roll.index, sh_roll.values, 0,
                     where=sh_roll.values < 0, alpha=0.2, color='#e74c3c')
    ax3.axhline(0, color='white', linewidth=1)
    ax3.axhline(sh_roll.mean(), color='yellow', linewidth=1,
                linestyle='--', label=f'Media: {sh_roll.mean():.2f}')
    ax3.set_ylabel('Sharpe Ratio')
    ax3.set_title('Sharpe Ratio rodante — Optimo Global (ventana 24 meses)\n'
                  'Verde=eficiente, Rojo=periodo dificil', fontsize=11)
    ax3.legend(fontsize=8)
    ax3.grid(alpha=0.15)

plt.tight_layout()
plt.savefig('outputs/21_rolling_metrics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Resumen rolling y beta vs SP500
resumen_r = resumen_rolling(retornos, portafolios, meta)
print('Resumen Sharpe rodante por portafolio (ventana 12 meses):')
df_r = resumen_r.copy()
for c in ['sharpe_medio','sharpe_min','sharpe_max','sharpe_actual']:
    df_r[c] = df_r[c].map('{:.3f}'.format)
df_r['pct_positivo'] = df_r['pct_positivo'].map('{:.0%}'.format)
display(df_r)

# Beta rodante del optimo vs SP500
if 'SP500' in retornos.columns and portafolios.get('optimo'):
    opt = portafolios['optimo']
    fondos = [f for f in opt['composicion'] if f in retornos.columns]
    w = np.array([opt['composicion'][f] for f in fondos])
    R = retornos[fondos].dropna()
    ret_port = pd.Series(R.values @ (w/w.sum()), index=R.index)

    beta_r = beta_rodante(ret_port, retornos['SP500'], ventana=12)

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(beta_r.index, beta_r.values, color='#FFD700', linewidth=2,
            label='Beta vs SP500 (12m)')
    ax.axhline(1.0, color='white', linewidth=1, linestyle='--', label='Beta = 1')
    ax.axhline(0.0, color='gray',  linewidth=1, linestyle=':',  label='Beta = 0')
    ax.fill_between(beta_r.index, beta_r.values, 1.0,
                    where=beta_r.values > 1, alpha=0.15, color='#e74c3c',
                    label='Mayor riesgo que SP500')
    ax.fill_between(beta_r.index, beta_r.values, 1.0,
                    where=beta_r.values < 1, alpha=0.15, color='#2ecc71',
                    label='Menor riesgo que SP500')
    ax.set_ylabel('Beta')
    ax.set_title('Beta rodante del Optimo Global vs SP500 (ventana 12 meses)', fontsize=11)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.15)
    plt.tight_layout()
    plt.savefig('outputs/22_beta_rodante.png', dpi=150, bbox_inches='tight')
    plt.show()


## 18. Stress Testing Hipotético <a id='18-stress-hipotetico'></a>

A diferencia del stress test histórico (sección 13), aquí definimos
**escenarios futuros que no han ocurrido** pero podrían ocurrir.

El impacto se calcula via las **sensibilidades** (betas) de cada fondo:
$$\Delta r_{portafolio} = \sum_i w_i \cdot \beta_i^{mercado} \cdot \Delta_{mercado} + w_i \cdot \beta_i^{tasa} \cdot \Delta_{tasa} + ...$$



In [ ]:
from src.stress_hipotetico import stress_hipotetico, calcular_sensibilidades, ESCENARIOS_HIPOTETICOS

# Calcular sensibilidades de cada fondo
print('Calculando sensibilidades (betas) de cada fondo...')
sensibilidades = calcular_sensibilidades(retornos, meta)

# Top 5 fondos mas sensibles al mercado
top_mercado = sensibilidades['beta_mercado'].sort_values(ascending=False)
print('\nFondos mas sensibles al mercado (beta_mercado):')
for fid, beta in top_mercado.head(5).items():
    nombre = meta.loc[fid,'nombre'] if fid in meta.index else fid
    print(f'  {nombre[:40]:40} beta={beta:.3f}')

print('\nFondos menos sensibles (defensivos):')
for fid, beta in top_mercado.tail(5).items():
    nombre = meta.loc[fid,'nombre'] if fid in meta.index else fid
    print(f'  {nombre[:40]:40} beta={beta:.3f}')


In [ ]:
# Stress test hipotetico
df_hip = stress_hipotetico(portafolios, retornos, meta, monto=MONTO)

# Mostrar tabla formateada (solo % de impacto)
cols_port = [c for c in ['Conservador','Moderado','Agresivo','S&P 500','Optimo Global']
             if c in df_hip.columns]
df_hip_show = df_hip[['Escenario'] + cols_port].copy()
for c in cols_port:
    df_hip_show[c] = df_hip_show[c].map('{:+.1%}'.format)
display(df_hip_show)


In [ ]:
# Heatmap stress hipotetico
cols_port = [c for c in ['Conservador','Moderado','Agresivo','S&P 500','Optimo Global']
             if c in df_hip.columns]
data_hip = df_hip[cols_port].values.astype(float)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Heatmap hipotetico
ax = axes[0]
im = ax.imshow(data_hip.T, cmap='RdYlGn', aspect='auto', vmin=-0.30, vmax=0.30)
ax.set_xticks(range(len(df_hip)))
ax.set_xticklabels(df_hip['Escenario'], fontsize=7, rotation=20, ha='right')
ax.set_yticks(range(len(cols_port)))
ax.set_yticklabels(cols_port, fontsize=9)
plt.colorbar(im, ax=ax, label='Impacto en retorno')
ax.set_title('Stress Test HIPOTETICO\n(escenarios futuros posibles)', fontsize=12)
for i in range(len(cols_port)):
    for j in range(len(df_hip)):
        val = data_hip[j, i]
        ax.text(j, i, f'{val:+.0%}', ha='center', va='center',
                fontsize=8, color='black' if abs(val) < 0.20 else 'white',
                fontweight='bold')

# Impacto en CLP del peor escenario (crisis sistemica)
ax2 = axes[1]
escenario_peor = 'Crisis sistemica'
if escenario_peor in df_hip['Escenario'].values:
    fila = df_hip[df_hip['Escenario'] == escenario_peor].iloc[0]
    valores_clp = [fila[c] * MONTO for c in cols_port if c in fila.index]
    colores_bar = ['#e74c3c' if v < 0 else '#2ecc71' for v in valores_clp]
    bars = ax2.bar(cols_port[:len(valores_clp)], [v/1e6 for v in valores_clp],
                   color=colores_bar, edgecolor='white')
    for bar, val in zip(bars, [v/1e6 for v in valores_clp]):
        ax2.text(bar.get_x()+bar.get_width()/2,
                 val + 0.05 * np.sign(val),
                 f'${val:.2f}M', ha='center', fontsize=9,
                 va='bottom' if val >= 0 else 'top')
    ax2.axhline(0, color='white', linewidth=1)
    ax2.set_ylabel('Impacto ($M CLP)')
    ax2.set_title(f'Escenario: {escenario_peor}\nImpacto en ${MONTO/1e6:.0f}M CLP', fontsize=11)
    ax2.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('outputs/23_stress_hipotetico.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nConclusiones del stress test hipotetico:')
print('  - Conservador: resistente a crashes de mercado, vulnerable a subida de tasas')
print('  - Agresivo/SP500: muy sensibles a caidas del mercado (-20% a -25%)')
print('  - Optimo Global: balance entre exposicion y proteccion gracias al 30% SP500')
